# 🌍 Enriquecimiento de Datos: Baseline Espacial, Contextual y NLP

## 🎯 Objetivo del Cuaderno
Este *notebook* consolida el *Master Dataset* (Baseline) que alimentará a los modelos predictivos. Su función es fusionar las características estructurales limpias con el valor semántico extraído por la IA y enriquecer el resultado con inteligencia geoespacial.

## ⚙️ Flujo de Transformación:
1. **Geocodificación:** Conversión de direcciones de texto en coordenadas espaciales (Latitud/Longitud) utilizando `Geopy`.
2. **Contexto Socioeconómico (Macro):** Inyección de la variable **Renta Neta Media por Distrito**, proporcionando al algoritmo un anclaje económico robusto del entorno.
3. **Procesamiento de Lenguaje Natural (NLP):** Análisis de sentimiento y extracción de palabras clave de las descripciones textuales para capturar características intangibles (ej. urgencia de venta, luminosidad).
4. **Fusión:** Integración final con el *dataframe* de métricas visuales procesadas por Google Gemini.

In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt

In [2]:
df = pd.read_parquet("../DATOS/imagenes_procesadas_total.parquet", engine='fastparquet')

In [3]:
df.iloc[:,:15]

,Enlace,Precio,Dormitorios,Superficie,Num_baños,Descripción,Metro,Renfe,Bus,Farmacia,Bar,Restaurante,Hospital,Supermercado,Tienda
0,https://www.tecnocasa.es/venta/piso/madrid/alc...,368000.0,4,131.0,2,AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES BARR...,0,1,1,1,1,1,1,1,1
1,https://www.tecnocasa.es/venta/casa/madrid/san...,331000.0,3,139.0,2,TECNOCASA SAN MARTIN DE LA VEGA- Vende viviend...,0,0,1,1,0,0,0,0,0
2,https://www.tecnocasa.es/venta/piso/madrid/arg...,304000.0,3,126.0,2,Agencia inmobiliaria de ARGANDA DEL REY-zona L...,1,0,1,1,1,1,0,1,1
3,https://www.tecnocasa.es/venta/casa/madrid/arg...,450000.0,4,255.0,2,Agencia inmobiliaria de ARGANDA DEL REY-zona L...,1,0,1,1,1,1,0,1,0
4,https://www.tecnocasa.es/venta/piso/madrid/mos...,175000.0,2,46.0,1,Agencia inmobiliaria de MÓSTOLES zona AVENIDA ...,1,1,1,1,1,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,https://www.tecnocasa.es/venta/piso/madrid/alc...,269900.0,2,136.0,2,"AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES, OFI...",0,1,1,1,1,1,0,1,1
1011,https://www.tecnocasa.es/venta/casa/madrid/alc...,395000.0,6,237.0,3,"AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES, OFI...",0,1,1,1,0,1,0,1,1
1012,https://www.tecnocasa.es/venta/casa/madrid/alc...,379900.0,4,176.0,1,"AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES, OFI...",0,1,1,1,0,1,0,1,1
1013,https://www.tecnocasa.es/venta/piso/madrid/poz...,520000.0,2,103.0,2,Agencia inmobiliaria de Pozuelo de Alarcón Est...,0,1,1,1,1,1,1,1,1


In [4]:
df.iloc[:,15:30]

,Colegio,Imagenes,Vistas,Estado_conservacion,Garaje,Trastero,Piscina,Terraza_balcon,Seguridad_portero,Etiqueta,Amueblado,Calle_tranquila,Imagenes_locales,Aire acondicionado,Tipo_de_inmueble
0,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Desconocido,0,0,1,0,0,Desconocida,0,1,"[imagenes_tecnocasa\anuncio_0\0.jpeg, imagenes...",1,Popular
1,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Desconocido,1,0,1,0,0,F,0,0,"[imagenes_tecnocasa\anuncio_1\0.jpeg, imagenes...",0,Media
2,1,[https://cdn-media.medialabtc.it/es/agencies/m...,1,Desconocido,1,1,0,1,0,Desconocida,0,0,"[imagenes_tecnocasa\anuncio_3\0.jpeg, imagenes...",1,Popular
3,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Desconocido,1,0,1,0,0,E,0,0,"[imagenes_tecnocasa\anuncio_4\0.jpeg, imagenes...",1,Popular
4,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Desconocido,0,0,0,1,0,B,0,0,"[imagenes_tecnocasa\anuncio_5\0.jpeg, imagenes...",0,Media
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Desconocido,1,1,0,0,0,Desconocida,0,0,"[imagenes_tecnocasa\anuncio_1635\0.jpeg, image...",1,Media
1011,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Buen estado,1,0,0,0,0,Desconocida,0,0,"[imagenes_tecnocasa\anuncio_1636\0.jpeg, image...",1,Señorial
1012,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Desconocido,1,0,1,1,0,E,0,0,"[imagenes_tecnocasa\anuncio_1637\0.jpeg, image...",1,Media
1013,1,[https://cdn-media.medialabtc.it/es/agencies/m...,0,Desconocido,1,1,0,0,0,B,0,0,"[imagenes_tecnocasa\anuncio_1650\0.jpeg, image...",1,Popular


In [5]:
df.iloc[:,30:45]

,Año_de_construccion,Calefaccion,Jardin,Ascensor,Tipo_vivienda,Planta_ordinal,distrito/ciudad,Localización_normalizada,Latitud,Longitud,announcement_id,Precio_m2,Longitud_desc,Riqueza_desc,Polaridad_desc
0,1991,0,0,0,piso,3,Alcalá de Henares,"Pryconsa - Juan De Austria - El Val, El Val, A...",40.481954,-3.363981,0,2809.160305,251.0,169.0,0.000000
1,1977,1,1,0,casa,2,San Martín de la Vega,San Martín De La Vega,40.211046,-3.577181,1,2381.294964,159.0,110.0,0.000000
2,1991,1,0,1,piso,4,Arganda del Rey,Arganda Del Rey,40.300771,-3.438069,3,2412.698413,249.0,166.0,0.333333
3,1991,1,0,0,casa,1,Arganda del Rey,Arganda Del Rey,40.300771,-3.438069,4,1764.705882,250.0,175.0,1.000000
4,1967,1,0,0,piso,1,Móstoles,Móstoles,40.323853,-3.864921,5,3804.347826,94.0,76.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,2000,1,0,0,piso,1,Alcalá de Henares,"Ensanche, Alcalá De Henares",40.481954,-3.363981,1635,1984.558824,219.0,154.0,0.050000
1011,1999,0,0,0,casa,1,Alcalá de Henares,"Ensanche, Alcalá De Henares",40.481954,-3.363981,1636,1666.666667,278.0,199.0,0.000000
1012,1999,1,0,0,casa,1,Alcalá de Henares,"Ensanche, Alcalá De Henares",40.481954,-3.363981,1637,2158.522727,274.0,196.0,0.000000
1013,2019,1,0,0,piso,2,Pozuelo de Alarcón,Pozuelo De Alarcón,40.434653,-3.814834,1650,5048.543689,170.0,134.0,0.050000


In [6]:
df.iloc[:,45:60]

,Subjetividad_desc,Adjetivos_positivos,Urgencia_desc,tipo_suelo,material_pared,tipo_ventana,altura_techo,elementos_estructurales_visibles,nivel_lujo,nivel_modernidad,estilo_principal,calidad_iluminacion,espacio_abierto,distribucion_espacios,espacios_almacenaje
0,0.40,1.0,0.0,parquet,pintura,corredera,estándar (2.4-2.8m),nada,3.6,3.8,convencional,buena,False,convencional,normal
1,0.00,1.0,0.0,cerámica,pintura,corredera,estándar (2.4-2.8m),nada,3.8,2.8,clásico,regular,False,convencional,limitado
2,0.60,2.0,1.0,cerámica,pintura,corredera,estándar (2.4-2.8m),nada,3.2,4.0,convencional,regular,False,convencional,normal
3,1.00,2.0,1.0,mármol,pintura,corredera,estándar (2.4-2.8m),nada,3.8,3.2,clásico,regular,False,convencional,limitado
4,0.25,0.0,0.0,cerámica,pintura,corredera,estándar (2.4-2.8m),nada,3.2,3.8,convencional,buena,False,convencional,limitado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,0.40,0.0,1.0,parquet,pintura,ninguna visible,estándar (2.4-2.8m),nada,3.8,4.4,convencional,regular,False,convencional,limitado
1011,0.00,1.0,0.0,otro,pintura,corredera,estándar (2.4-2.8m),nada,3.4,4.6,convencional,regular,False,convencional,limitado
1012,0.00,2.0,0.0,parquet,pintura,corredera,estándar (2.4-2.8m),nada,4.2,4.0,clásico,buena,False,convencional,normal
1013,0.40,0.0,0.0,parquet,pintura,ninguna visible,estándar (2.4-2.8m),nada,4.8,5.8,moderno,regular,False,convencional,limitado


In [7]:
df.iloc[:,60:]

,zonas_exteriores,tecnologia_visible,calidad_bano,equipamiento_cocina,defectos_visibles,nivel_foto_profesional,equilibrio_composicion,distorsion_perspectiva,num_imagenes,color_diversidad,color_avg_hue,color_avg_saturacion,color_avg_valor,color_principal
0,ninguno,ninguno,estándar,básica,ninguno,7.6,6.6,ninguna,5,3,0.198413,0.444444,0.775163,#FFFFFF
1,ninguno,ninguno,estándar,básica,ninguno,6.4,5.6,ninguna,5,3,0.184541,0.525395,0.711111,#228B22
2,ninguno,ninguno,estándar,básica,ninguno,6.4,6.0,ninguna,5,3,0.057181,0.276780,0.809150,#A77A54
3,ninguno,ninguno,estándar,básica,ninguno,6.0,7.0,ninguna,5,3,0.096578,0.561367,0.711111,#8B4513
4,ninguno,ninguno,estándar,parcialmente equipada,ninguno,7.4,6.4,ninguna,5,3,0.131290,0.210396,0.943791,#F5F5DC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,ninguno,ninguno,estándar,básica,ninguno,4.8,6.0,ninguna,5,3,0.104720,0.404228,0.819608,#DEB887
1011,ninguno,ninguno,estándar,básica,ninguno,6.4,6.6,ninguna,5,3,0.195425,0.333333,0.833987,#808080
1012,ninguno,ninguno,estándar,parcialmente equipada,ninguno,6.2,7.2,ninguna,5,3,0.049770,0.212812,0.760784,#A08E7B
1013,ninguno,ninguno,estándar,básica,ninguno,6.8,6.6,ninguna,5,3,0.085467,0.573105,0.724183,#8B4513


## 📡 Análisis geoespacial de exposición al ruido ambiental

Este bloque de código utiliza **GeoPandas** para calcular la exposición al ruido de cada inmueble según su proximidad a diferentes fuentes de ruido en la Comunidad de Madrid. Las fuentes de ruido consideradas provienen de shapefiles oficiales (formato .shp) y se relacionan espacialmente con cada anuncio inmobiliario.

#### Funcionalidad principal:

1. **Conversión a GeoDataFrame**:
   - Transforma el DataFrame en un `GeoDataFrame` usando la latitud y longitud de cada anuncio.

2. **Carga de shapefiles**:
   - Se importan capas espaciales de ruido por carreteras, líneas ferroviarias y aglomeraciones urbanas.

3. **Reproyección**:
   - Se transforman todos los datos a un sistema métrico (EPSG:25830 - UTM zona 30N) para calcular distancias en metros con precisión.

4. **Join espacial (más cercano)**:
   - Para cada anuncio, se calcula la distancia al polígono de ruido más cercano en cada categoría (carreteras, ferrocarril, aglomeraciones).
   - Se extrae el valor `Lden` (índice de ruido en decibelios ponderado por tiempo) y se almacena en nuevas columnas: `ruido_carretera`, `ruido_ferrocarril` y `ruido_urbano`.

5. **Aplicación de umbrales de distancia**:
   - Si un inmueble está demasiado lejos de una fuente de ruido (ej. >1000 m de una carretera), se descarta ese valor como influencia.

6. **Cálculo del ruido promedio**:
   - Se calcula el `ruido_promedio` usando solo los valores válidos disponibles para cada inmueble.

7. **Reproyección final**:
   - Se transforma nuevamente el `GeoDataFrame` a coordenadas geográficas estándar (EPSG:4326) para su exportación o visualización en herramientas como folium, Kepler.gl o QGIS.

> ℹ️ Esta etapa es clave para incorporar variables ambientales en modelos de valoración o segmentación inmobiliaria.


In [9]:
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

# Crear GeoDataFrame desde df
df = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Longitud, df.Latitud), crs="EPSG:4326")

# Cargar shapefiles
ruido_carreteras = gpd.read_file("../DATOS/mer-carreteras_lden/CDERUI02_Lden_RepGeo_LocalId_C.shp")
ruido_ferroviario = gpd.read_file("../DATOS/mer-lineas_ferroviarias_lden/CDERUI02_Lden_RepGeo_LocalId_F.shp")
ruido_aglomeraciones = gpd.read_file("../DATOS/mer-aglomeraciones_lden/CDERUI02_Lden_RepGeo_LocalId_Ag.shp")
ruido_aeropuertos = gpd.read_file("../DATOS/mer-aeropuertos_lden/CDERUI02_Lden_RepGeo_LocalId_A.shp")

# Reproyectar a sistema métrico (UTM zona 30N)
proj_crs = "EPSG:25830"
df = df.to_crs(proj_crs)
ruido_carreteras = ruido_carreteras.to_crs(proj_crs)
ruido_ferroviario = ruido_ferroviario.to_crs(proj_crs)
ruido_aglomeraciones = ruido_aglomeraciones.to_crs(proj_crs)
ruido_aeropuertos = ruido_aeropuertos.to_crs(proj_crs)

# ---- Aplicar umbrales de distancia
umbral_carretera = 1000     # metros
umbral_ferrocarril = 1500   # metros
umbral_aglomeracion = 1500  # metros
umbral_aeropuerto = 3000    # metros

# ---- JOIN NEAREST 1: Carreteras
temp = df.sjoin_nearest(
    ruido_carreteras[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_carretera',
    max_distance=umbral_carretera
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_carretera'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 2: Ferroviario
temp = df.sjoin_nearest(
    ruido_ferroviario[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_ferrocarril',
    max_distance=umbral_ferrocarril
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_ferrocarril'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 3: Aglomeraciones
temp = df.sjoin_nearest(
    ruido_aglomeraciones[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_aglomeracion',
    max_distance=umbral_aglomeracion
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_urbano'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 4: Aeropuertos
temp = df.sjoin_nearest(
    ruido_aeropuertos[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_aeropuerto',
    max_distance=umbral_aeropuerto
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_aeropuerto'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')


df.loc[df['dist_a_carretera'] > umbral_carretera, 'ruido_carretera'] = np.nan
df.loc[df['dist_a_ferrocarril'] > umbral_ferrocarril, 'ruido_ferrocarril'] = np.nan
df.loc[df['dist_a_aglomeracion'] > umbral_aglomeracion, 'ruido_urbano'] = np.nan
df.loc[df['dist_a_aeropuerto'] > umbral_aeropuerto, 'ruido_aeropuerto'] = np.nan

# ---- Calcular promedio solo con fuentes válidas
df['ruido_promedio'] = df[['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']].mean(axis=1)

# ---- Reproyectar a WGS84 para exportar/visualizar
df = df.to_crs("EPSG:4326")

In [10]:
df.iloc[:,-10:]

,geometry,ruido_carretera,dist_a_carretera,ruido_ferrocarril,dist_a_ferrocarril,ruido_urbano,dist_a_aglomeracion,ruido_aeropuerto,dist_a_aeropuerto,ruido_promedio
0,POINT (-3.36398 40.48195),NaN,NaN,NaN,NaN,65.0,0.118166,NaN,NaN,65.0
1,POINT (-3.57718 40.21105),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,POINT (-3.43807 40.30077),NaN,NaN,55.0,1089.776576,NaN,NaN,NaN,NaN,55.0
3,POINT (-3.43807 40.30077),NaN,NaN,55.0,1089.776576,NaN,NaN,NaN,NaN,55.0
4,POINT (-3.86492 40.32385),50.0,764.722856,NaN,NaN,NaN,NaN,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
1010,POINT (-3.36398 40.48195),NaN,NaN,NaN,NaN,65.0,0.118166,NaN,NaN,65.0
1011,POINT (-3.36398 40.48195),NaN,NaN,NaN,NaN,65.0,0.118166,NaN,NaN,65.0
1012,POINT (-3.36398 40.48195),NaN,NaN,NaN,NaN,65.0,0.118166,NaN,NaN,65.0
1013,POINT (-3.81483 40.43465),50.0,148.344079,55.0,688.211151,NaN,NaN,NaN,NaN,52.5


In [13]:
# Definir el valor mínimo de ruido ambiental
ruido_minimo = 30

# Reemplazar NaN y 0.0 en cada columna de ruido
for col in ['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']:
    df[col] = df[col].replace(0.0, np.nan)
    df[col].fillna(ruido_minimo)

# Recalcular el promedio
df['ruido_promedio'] = df[['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']].mean(axis=1)
df['ruido_promedio'].fillna(ruido_minimo)

# Asignar un valor mayor que el umbral cuando no hay fuente cercana
df['dist_a_carretera'] = df['dist_a_carretera'].fillna(umbral_carretera + 100)
df['dist_a_ferrocarril'] = df['dist_a_ferrocarril'].fillna(umbral_ferrocarril + 100)
df['dist_a_aglomeracion'] = df['dist_a_aglomeracion'].fillna(umbral_aglomeracion + 100)
df['dist_a_aeropuerto'] = df['dist_a_aeropuerto'].fillna(umbral_aeropuerto + 100)

df.iloc[:,-10:]

,geometry,ruido_carretera,dist_a_carretera,ruido_ferrocarril,dist_a_ferrocarril,ruido_urbano,dist_a_aglomeracion,ruido_aeropuerto,dist_a_aeropuerto,ruido_promedio
0,POINT (-3.36398 40.48195),30.0,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75
1,POINT (-3.57718 40.21105),30.0,1100.000000,30.0,1600.000000,30.0,1600.000000,30.0,3100.0,30.00
2,POINT (-3.43807 40.30077),30.0,1100.000000,55.0,1089.776576,30.0,1600.000000,30.0,3100.0,36.25
3,POINT (-3.43807 40.30077),30.0,1100.000000,55.0,1089.776576,30.0,1600.000000,30.0,3100.0,36.25
4,POINT (-3.86492 40.32385),50.0,764.722856,30.0,1600.000000,30.0,1600.000000,30.0,3100.0,35.00
...,...,...,...,...,...,...,...,...,...,...
1010,POINT (-3.36398 40.48195),30.0,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75
1011,POINT (-3.36398 40.48195),30.0,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75
1012,POINT (-3.36398 40.48195),30.0,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75
1013,POINT (-3.81483 40.43465),50.0,148.344079,55.0,688.211151,30.0,1600.000000,30.0,3100.0,41.25


## 💰 Enriquecimiento sociodemográfico con datos del INE: renta neta media por hogar

En esta sección se incorporan **datos de renta neta media por hogar** proporcionados por el Instituto Nacional de Estadística (INE), con el objetivo de añadir una capa sociodemográfica al análisis inmobiliario.

#### Fuente:
- Indicadores de renta media y mediana por municipio y distrito (año 2022).

#### Proceso realizado:

1. **Limpieza y normalización**:
   - Se extraen los nombres de municipios a partir del campo codificado del INE (`"28001 Acebeda, La"` → `"ACEBEDA, LA"`).
   - Se identifican y renombran los distritos de la ciudad de Madrid a partir del código oficial (ej. `"2807908"` → `"FUENCARRAL-EL PARDO"`).

2. **Clave de emparejamiento**:
   - Se crea un campo unificado `Localizacion_Clave` que contiene el nombre del **municipio** o del **distrito** si aplica, todo en mayúsculas y limpio.

3. **Enriquecimiento del dataset principal**:
   - A partir del nombre del municipio o distrito, se asigna el valor de `renta_neta_media_hogar` a cada inmueble del dataset.
   - Esto permite analizar la relación entre el contexto económico y las características del entorno, como el nivel de ruido o los precios de los inmuebles.

> 🧠 Esta capa de datos añade un importante contexto social que puede ser utilizado en modelos predictivos, análisis de accesibilidad o estudios de desigualdad territorial.


In [12]:
df_rentas = pd.read_csv("../DATOS/rentaMedia.csv", sep = ";")
df_rentas

,Municipios,Distritos,Secciones,Indicadores de renta media y mediana,Periodo,Total
0,28001 La Acebeda,NaN,NaN,Renta neta media por hogar,2022,29.297
1,28002 Ajalvir,NaN,NaN,Renta neta media por hogar,2022,43.032
2,28003 Alameda del Valle,NaN,NaN,Renta neta media por hogar,2022,33.055
3,28004 El Alamo,NaN,NaN,Renta neta media por hogar,2022,34.027
4,28005 Alcala de Henares,NaN,NaN,Renta neta media por hogar,2022,39.439
...,...,...,...,...,...,...
195,28182 Villavieja del Lozoya,NaN,NaN,Renta neta media por hogar,2022,31.716
196,28183 Zarzalejo,NaN,NaN,Renta neta media por hogar,2022,30.057
197,28901 Lozoyuela-Navas-Sieteiglesias,NaN,NaN,Renta neta media por hogar,2022,31.563
198,28902 Puentes Viejas,NaN,NaN,Renta neta media por hogar,2022,34.732


In [49]:
df_rentas = df_rentas.copy()

# Extraer nombre de municipio limpio (sin código)
df_rentas["Municipio_Limpio"] = df_rentas["Municipios"].str.extract(r"\d+\s+(.*)")
df_rentas["Municipio_Limpio"] = df_rentas["Municipio_Limpio"].str.upper().str.strip()

# Extraer distritos de Madrid (codificados como 28079 Madrid distrito XX)
df_rentas["Distrito_Codigo"] = df_rentas["Distritos"].str.extract(r"(28079\d{2})")

# Crear tabla de equivalencia: código → nombre de distrito
codigos_distritos = {
    "2807901": "CENTRO",
    "2807902": "ARGANZUELA",
    "2807903": "RETIRO",
    "2807904": "SALAMANCA",
    "2807905": "CHAMARTIN",
    "2807906": "TETUAN",
    "2807907": "CHAMBERI",
    "2807908": "FUENCARRAL-EL PARDO",
    "2807909": "MONCLOA-ARAVACA",
    "2807910": "LATINA",
    "2807911": "CARABANCHEL",
    "2807912": "USERA",
    "2807913": "PUENTE DE VALLECAS",
    "2807914": "MORATALAZ",
    "2807915": "CIUDAD LINEAL",
    "2807916": "HORTALEZA",
    "2807917": "VILLAVERDE",
    "2807918": "VILLA DE VALLECAS",
    "2807919": "VICALVARO",
    "2807920": "SAN BLAS-CANILLEJAS",
    "2807921": "BARAJAS"
}

# Asignar nombres de distritos
df_rentas["Distrito_Limpio"] = df_rentas["Distrito_Codigo"].map(codigos_distritos)

# Llenar campo "Localizacion_Clave" con MUNICIPIO o DISTRITO limpio
df_rentas["Localizacion_Clave"] = df_rentas["Distrito_Limpio"].fillna(df_rentas["Municipio_Limpio"])


In [50]:
df_rentas.iloc[77:99,:]

,Municipios,Distritos,Secciones,Indicadores de renta media y mediana,Periodo,Total,Municipio_Limpio,Distrito_Codigo,Distrito_Limpio,Localizacion_Clave
77,28079 Madrid,NaN,NaN,Renta neta media por hogar,2022,46.651,MADRID,NaN,NaN,MADRID
78,28079 Madrid,2807901 Madrid distrito 01,NaN,Renta neta media por hogar,2022,41.059,MADRID,2807901,CENTRO,CENTRO
79,28079 Madrid,2807902 Madrid distrito 02,NaN,Renta neta media por hogar,2022,49.236,MADRID,2807902,ARGANZUELA,ARGANZUELA
80,28079 Madrid,2807903 Madrid distrito 03,NaN,Renta neta media por hogar,2022,60.762,MADRID,2807903,RETIRO,RETIRO
81,28079 Madrid,2807904 Madrid distrito 04,NaN,Renta neta media por hogar,2022,64.925,MADRID,2807904,SALAMANCA,SALAMANCA
82,28079 Madrid,2807905 Madrid distrito 05,NaN,Renta neta media por hogar,2022,74.842,MADRID,2807905,CHAMARTIN,CHAMARTIN
83,28079 Madrid,2807906 Madrid distrito 06,NaN,Renta neta media por hogar,2022,42.465,MADRID,2807906,TETUAN,TETUAN
84,28079 Madrid,2807907 Madrid distrito 07,NaN,Renta neta media por hogar,2022,60.032,MADRID,2807907,CHAMBERI,CHAMBERI
85,28079 Madrid,2807908 Madrid distrito 08,NaN,Renta neta media por hogar,2022,58.214,MADRID,2807908,FUENCARRAL-EL PARDO,FUENCARRAL-EL PARDO
86,28079 Madrid,2807909 Madrid distrito 09,NaN,Renta neta media por hogar,2022,68.238,MADRID,2807909,MONCLOA-ARAVACA,MONCLOA-ARAVACA


In [51]:
import pandas as pd
import unidecode

# Lista de distritos de Madrid capital
distritos_madrid = [
    "CENTRO", "ARGANZUELA", "RETIRO", "SALAMANCA", "CHAMARTIN", "TETUAN", "CHAMBERI",
    "FUENCARRAL-EL PARDO", "MONCLOA-ARAVACA", "LATINA", "CARABANCHEL", "USERA",
    "PUENTE DE VALLECAS", "MORATALAZ", "CIUDAD LINEAL", "HORTALEZA",
    "VILLAVERDE", "VILLA DE VALLECAS", "VICALVARO", "SAN BLAS-CANILLEJAS", "BARAJAS"
]

# Mapeo de barrios a distritos (ejemplo parcial, puedes ampliarlo)
barrios_a_distritos = {
    "CASTILLEJOS": "TETUAN",
    "BERRUGUETE": "TETUAN",
    "JUSTICIA": "CENTRO",
    "UNIVERSIDAD": "CENTRO",
    "LUCERO": "LATINA",
    "BELLAS VISTAS": "TETUAN",
    "FUENCARRAL": "FUENCARRAL-EL PARDO",
    "EMBAJADORES": "CENTRO",
    "ALMENDRALES": "USERA",
    "ARAVACA": "MONCLOA-ARAVACA",
    "ARROYO DEL FRESNO": "FUENCARRAL-EL PARDO",
    "FUENTE DEL BERRO": "SALAMANCA",
    "ARGUELLES": "MONCLOA-ARAVACA",
    "LISTA": "SALAMANCA",
    "ALUCHE": "LATINA",
    "CUATRO VIENTOS": "LATINA",
    "GUINDALERA": "SALAMANCA",
    "ENTREVIAS": "PUENTE DE VALLECAS",
    "CIUDAD JARDIN": "CHAMARTIN",
    "ACACIAS": "ARGANZUELA",
    "LOS SANTOS DE LA HUMOSA": "LOS SANTOS DE LA HUMOSA"
    # Añade más si es necesario
}

def limpiar_localizacion(valor):
    if pd.isna(valor):
        return None
    val = unidecode.unidecode(valor.upper().strip())
    if val in distritos_madrid:
        return val
    if val in barrios_a_distritos:
        return barrios_a_distritos[val]
    return val  # Asumimos que es un municipio

# Crear la nueva columna
df["Localizacion_Clave"] = df["distrito/ciudad"].apply(limpiar_localizacion)

df = df.merge(
    df_rentas[["Localizacion_Clave", "Total"]],
    on="Localizacion_Clave",
    how="left"
)

df.rename(columns={"Total": "Renta_Neta_Media"}, inplace=True)

In [52]:
# Comprobar si algún distrito o municpio quedó fuera
df[df["Renta_Neta_Media"].isna()]["distrito/ciudad"].value_counts()

Series([], Name: count, dtype: int64)

In [53]:
df.iloc[:,-2:]

,Localizacion_Clave,Renta_Neta_Media
0,ALCALA DE HENARES,39.439
1,SAN MARTIN DE LA VEGA,35.818
2,ARGANDA DEL REY,37.574
3,ARGANDA DEL REY,37.574
4,MOSTOLES,36.297
...,...,...
1010,ALCALA DE HENARES,39.439
1011,ALCALA DE HENARES,39.439
1012,ALCALA DE HENARES,39.439
1013,POZUELO DE ALARCON,92.596


In [54]:
#df.to_parquet("../DATOS/tecnocasa_final.parquet", index=False)